In [1]:
# ============================================================
# CELL 1 — Mount Drive & Install dependencies
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install -q torch torchvision timm
# DINOv2 is available via torch.hub directly from Meta

Mounted at /content/drive


In [2]:
# ============================================================
# CELL 2 — Configuration
# ============================================================
import os

BASE_DATASET = '/content/drive/MyDrive/4th year/S2/RL/Project/Dataset'

# All three dataset folders (matching your screenshot)
FRAMES_ROOTS = {
    # 'Activity':  os.path.join(BASE_DATASET, 'Activity/frames'),
    # 'Animals':   os.path.join(BASE_DATASET, 'Animals/frames'),
    'Incidents': os.path.join(BASE_DATASET, 'Incidents/frames'),
}

EMBEDDINGS_ROOT = os.path.join(BASE_DATASET, 'embeddings')
os.makedirs(EMBEDDINGS_ROOT, exist_ok=True)

BATCH_SIZE = 64   # frames per GPU batch — lower if you get OOM
DEVICE = 'cuda'   # Colab T4/A100 — will fall back to cpu automatically

In [3]:
# ============================================================
# CELL 3 — Load DINOv2 ViT-L/14 (dim=1024)
# ============================================================
import torch

device = torch.device(DEVICE if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load from Meta's torch.hub — downloads ~1.2 GB weights once
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')
model.eval().to(device)

print(f"✓ DINOv2 ViT-L/14 loaded")
print(f"  Output embedding dim: 1024")
print(f"  Parameters: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

Using device: cuda
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain.pth


100%|██████████| 1.13G/1.13G [00:07<00:00, 168MB/s] 


✓ DINOv2 ViT-L/14 loaded
  Output embedding dim: 1024
  Parameters: 304M


In [4]:
# ============================================================
# CELL 4 — Define preprocessing & embedding extraction
# ============================================================
import torch
import numpy as np
from torchvision import transforms
from PIL import Image
import glob

# DINOv2 official preprocessing
#   - Resize shortest edge to 518 (ViT-L/14 optimal input)
#   - Center crop to 518×518
#   - Normalize with ImageNet mean/std
preprocess = transforms.Compose([
    transforms.Resize(518, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(518),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def load_image_batch(paths):
    """Load a list of image paths into a batched tensor."""
    tensors = []
    valid_paths = []
    for p in paths:
        try:
            img = Image.open(p).convert('RGB')
            tensors.append(preprocess(img))
            valid_paths.append(p)
        except Exception as e:
            print(f"  ⚠ Skipping {os.path.basename(p)}: {e}")
    if not tensors:
        return None, []
    return torch.stack(tensors), valid_paths

@torch.no_grad()
def extract_embeddings_for_video(frame_dir, batch_size=64):
    """
    Given a folder of frames (frame_0001.jpg, ...),
    returns:
        embeddings : np.ndarray of shape (N, 1024)
        frame_names: list of N filenames (sorted)
    """
    frame_paths = sorted(glob.glob(os.path.join(frame_dir, "*.jpg")))
    if not frame_paths:
        return None, []

    all_embeddings = []
    all_names = []

    for i in range(0, len(frame_paths), batch_size):
        batch_paths = frame_paths[i:i+batch_size]
        batch_tensor, valid_paths = load_image_batch(batch_paths)
        if batch_tensor is None:
            continue

        batch_tensor = batch_tensor.to(device)

        # Forward pass — CLS token = [batch, 1024]
        embeddings = model(batch_tensor)          # shape: (B, 1024)
        all_embeddings.append(embeddings.cpu().numpy())
        all_names.extend([os.path.basename(p) for p in valid_paths])

    if not all_embeddings:
        return None, []

    return np.vstack(all_embeddings), all_names  # (N, 1024)

In [ ]:
# ============================================================
# CELL 5 — Main loop: iterate all folders, save .npz per video
# ============================================================
import time

total_videos = 0
total_frames = 0
skipped = 0

for category, frames_root in FRAMES_ROOTS.items():
    if not os.path.exists(frames_root):
        print(f"\n⚠ Frames folder not found, skipping: {frames_root}")
        continue

    print(f"\n{'='*60}")
    print(f"  Category: {category}")
    print(f"  Frames root: {frames_root}")
    print(f"{'='*60}")

    # Each sub-folder = one video
    video_dirs = sorted([
        d for d in os.listdir(frames_root)
        if os.path.isdir(os.path.join(frames_root, d))
    ])

    print(f"  Found {len(video_dirs)} video folders\n")

    # Mirror folder structure: embeddings/<Category>/<video_name>.npz
    category_emb_dir = os.path.join(EMBEDDINGS_ROOT, category)
    os.makedirs(category_emb_dir, exist_ok=True)

    for video_name in video_dirs:
        out_path = os.path.join(category_emb_dir, f"{video_name}.npz")

        # Skip if already computed
        if os.path.exists(out_path):
            existing = np.load(out_path)
            skipped += 1
            print(f"  [SKIP] {video_name} ({existing['embeddings'].shape[0]} frames already embedded)")
            continue

        frame_dir = os.path.join(frames_root, video_name)
        t0 = time.time()

        embeddings, frame_names = extract_embeddings_for_video(frame_dir, BATCH_SIZE)

        if embeddings is None:
            print(f"  [SKIP] {video_name} — no frames found")
            continue

        # Save as compressed numpy archive
        # Load with: data = np.load('x.npz'); embs = data['embeddings']
        np.savez_compressed(
            out_path,
            embeddings=embeddings,       # shape (N, 1024)  float32
            frame_names=frame_names,     # list of N filenames
            category=category,
            video_name=video_name,
        )

        elapsed = time.time() - t0
        total_videos += 1
        total_frames += len(frame_names)
        print(f"  ✓ {video_name}: {len(frame_names)} frames → (N,1024) | {elapsed:.1f}s")

print(f"\n{'='*60}")
print(f"DONE")
print(f"  Videos processed : {total_videos}")
print(f"  Videos skipped   : {skipped} (already existed)")
print(f"  Total frames embedded: {total_frames:,}")
print(f"  Embeddings saved to  : {EMBEDDINGS_ROOT}")


  Category: Activity
  Frames root: /content/drive/MyDrive/4th year/S2/RL/Project/Dataset/Activity/frames
  Found 77 video folders

  ✓ Archery__UCOn2HkJJt8: 370 frames → (N,1024) | 145.1s
  ✓ Arm_wrestling__A1EflBqBv14: 194 frames → (N,1024) | 82.9s
  ✓ Assembling_bicycle__BfnM0eyjB5Q: 371 frames → (N,1024) | 156.1s
  ✓ BMX__6uhLrPgbpUA: 92 frames → (N,1024) | 111.4s
  ✓ Baking_cookies__IGcsVPa34Hc: 429 frames → (N,1024) | 178.0s
  ✓ Ballet__c_NlYvL96y0: 250 frames → (N,1024) | 104.5s
  ✓ Bathing_dog__V90aT-d_FKo: 156 frames → (N,1024) | 66.4s
  ✓ Baton_twirling__dAa10hlgxCY: 457 frames → (N,1024) | 190.1s
  ✓ Beach_soccer__IoGpS8NQklE: 464 frames → (N,1024) | 193.6s
  ✓ Beer_pong__JDg--pjY5gg: 252 frames → (N,1024) | 106.5s
  ✓ Belly_dance__3K62qZ2hGyw: 257 frames → (N,1024) | 108.5s
  ✓ Blow-drying_hair__q6sLCLnTuik: 241 frames → (N,1024) | 101.6s
  ✓ Braiding_hair__qogdv5DWzkQ: 443 frames → (N,1024) | 185.3s
  ✓ Breakdancing__Jifw8dC5yTM: 300 frames → (N,1024) | 124.8s
  ✓ Brushin

In [ ]:
# ============================================================
# CELL 6 — Sanity check: load and inspect one embedding file
# ============================================================
import numpy as np
import os

category_emb_dir = os.path.join(EMBEDDINGS_ROOT, 'Incidents')
sample_file = sorted(os.listdir(category_emb_dir))[0]

data = np.load(os.path.join(category_emb_dir, sample_file))

print(f"File       : {sample_file}")
print(f"Embeddings : {data['embeddings'].shape}  (frames × 1024)")
print(f"dtype      : {data['embeddings'].dtype}")
print(f"Frames     : {list(data['frame_names'])[:5]} ...")
print(f"Category   : {data['category']}")
print(f"\nFirst embedding vector (first 8 dims):")
print(data['embeddings'][0, :8])